# Notebook 9 - MultiUAV Validation-Placement Study

This is the active text-first research entry point. It validates source provenance, the frozen session split, task eligibility, alias selection, and component wiring. It does not run the superseded Shepherd diagnostic workflow or claim revised-study model results.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import tempfile

BRANCH = 'codex/multiuav-validation-study'
IN_COLAB = bool(os.environ.get('COLAB_RELEASE_TAG'))
REPO = Path('/content/shepherd-ai') if IN_COLAB else Path.cwd()
if IN_COLAB and not (REPO / 'pyproject.toml').exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, 'https://github.com/cyberuniversal/shepherd-ai.git', str(REPO)], check=True)
elif IN_COLAB:
    subprocess.run(['git', '-C', str(REPO), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)
if not (REPO / 'pyproject.toml').exists():
    raise FileNotFoundError('Open this notebook from the Shepherd-AI repository.')
%cd {REPO}
%pip install -q -e .
src_path = str(REPO / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)


## 1. Audit the active component wiring

This verifies the committed source, split, and eligibility chain. It also records that Whisper, DistilBERT, the Week 8 deterministic pipeline, and the historical Week 9 diagnostic are not invoked by the primary revised study.

In [ ]:
from shepherd_ai.multiuav_study_wiring import audit_primary_study_wiring

wiring = audit_primary_study_wiring(REPO)
if not wiring['valid']:
    raise RuntimeError(json.dumps(wiring['errors'], indent=2))
print(json.dumps({
    'status': wiring['status'],
    'completed_gates': wiring['completed_gates'],
    'blocking_gates': wiring['blocking_gates'],
    'runtime_invocation': wiring['runtime_invocation'],
    'claim_status': wiring['claim_status'],
}, indent=2))


## 2. Inspect the frozen task population

The prospective five-case counts remain upper bounds. No intervention text or labels exist yet.

In [ ]:
metadata = REPO / 'datasets' / 'multiuav_plat'
source_audit = json.loads((metadata / 'source_audit_v1.json').read_text(encoding='utf-8'))
session_split = json.loads((metadata / 'session_split_v1.json').read_text(encoding='utf-8'))
eligibility = json.loads((metadata / 'task_eligibility_v1.json').read_text(encoding='utf-8'))
print(json.dumps({
    'source_counts': source_audit['benchmark']['counts'],
    'session_counts': session_split['session_counts'],
    'eligibility_summary': eligibility['summary'],
}, indent=2, sort_keys=True))


## 3. Optional source re-audit

Enable this CPU-only cell to clone the exact upstream commit and regenerate temporary source, split, and eligibility records. It does not overwrite committed research artifacts.

In [ ]:
RUN_SOURCE_REAUDIT = False
if RUN_SOURCE_REAUDIT:
    upstream = REPO / 'external' / 'MultiUAV-Plat'
    if not upstream.exists():
        subprocess.run(['git', 'clone', 'https://github.com/zhangsheng93/MultiUAV-Plat.git', str(upstream)], check=True)
    subprocess.run(['git', '-C', str(upstream), 'checkout', '1794e45e421fb5de03094f0b63f9ca95f86ab42f'], check=True)
    audit_dir = Path(tempfile.mkdtemp(prefix='shepherd-multiuav-audit-'))
    archive = upstream / 'benchmark' / 'benchmark.zip'
    subprocess.run(['python', 'scripts/audit_multiuav_plat_source.py', '--repository-root', str(upstream), '--output', str(audit_dir / 'source.json')], check=True)
    subprocess.run(['python', 'scripts/build_multiuav_session_split.py', '--archive', str(archive), '--output', str(audit_dir / 'split.json')], check=True)
    subprocess.run(['python', 'scripts/build_multiuav_task_eligibility.py', '--archive', str(archive), '--session-split', str(audit_dir / 'split.json'), '--output', str(audit_dir / 'eligibility.json')], check=True)
    print(audit_dir)
else:
    print('Source re-audit skipped. Set RUN_SOURCE_REAUDIT = True to reproduce it.')


## 4. Current stop condition

Do not run revised-study model inference yet. The next implemented gate must define AGENT-visible context and distinguish recoverable facts from information that genuinely requires operator clarification.

In [ ]:
assert wiring['ready_for_intervention_generation'] is False
assert wiring['ready_for_model_inference'] is False
print('Blocked gates:')
for gate in wiring['blocking_gates']:
    print('-', gate)
